### Master of Applied Artificial Intelligence

**Course: TC5035 - Proyecto Integrador**

<img src="https://github.com/Medicenchapin/Proyecto-Integrador/blob/main/assets/logo.png?raw=1" alt="Image Alt Text" width="500"/>


**Feature engineering**

Tutor: Dr. Horario Martinez Alfaro


Team members:
* Ignacio Jose Aguilar Garcia - A00819762
* Alejandro Calderon Aguilar - A01795353
* Ricardo Mar Cupido - A01795394

# FE

The model contains an individualized FE:

Each row represents a client (or case).

Within each row, in the drivers column, you have something like:

```bash
[
  {"feature": "arpu_90_days", "value": 123.4, "impact": 0.05},
  {"feature": "contacts", "value": 3, "impact": -0.03},
  {"feature": "plan_postpaid", "value": 1, "impact": 0.02},
  ...
]
```

That is, each row has its own important features with different weights.
👉 That's why we can't talk about the same set of global top features at the client level,
because SHAP tells what was most relevant for that particular prediction.

In [63]:
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import importlib, scripts.helpers as hp
importlib.reload(hp)
from scripts.helpers import Helpers

In [64]:
df = pd.read_parquet("../data/campaign_candidates_final.parquet")
df.head()

,state_name,previous_classification,previous_calls,client_age,network_age_years,banking,arpu_90_days,minutes_in,validity_average,average_performance,...,plan_postpaid,sn_banking,digital_index_mean,connected_days,charged_days,apps_days,music_gb,proba,sample_idx,drivers
0,GUATEMALA,NEW CLIENT,0,28.0,3.90,1,214.18,27.93,20.23,0.46,...,0.67,1.00,0.00,91,91,0,5.946,0.699237,0,"[{'feature': 'contacts', 'impact': -0.58204424..."
1,GUATEMALA,NEW CLIENT,0,19.0,0.66,0,92.58,48.77,3.14,0.61,...,0.75,1.00,0.00,76,55,4,0.031,0.781733,10,"[{'feature': 'contacts', 'impact': -0.43256592..."
2,SAN MARCOS,NEW CLIENT,0,20.0,0.34,1,175.49,83.58,4.00,0.25,...,0.12,0.56,0.00,91,59,30,0.000,0.641149,12,"[{'feature': 'client_age', 'impact': 0.4716295..."
3,SAN MARCOS,NEW CLIENT,0,NaN,1.90,1,97.54,51.89,8.36,0.77,...,0.54,0.38,0.29,90,77,11,0.000,0.690091,26,"[{'feature': 'plan_postpaid', 'impact': 0.5216..."
4,SAN MARCOS,NOT EFFECTIVE,2,31.0,5.87,1,102.25,78.93,15.00,0.69,...,0.24,0.30,0.00,89,64,22,0.000,0.733312,37,"[{'feature': 'contacts', 'impact': 0.383831799..."


In [65]:
helpers = Helpers(df=df)

# Get feature playbook

In [66]:
helpers.get_feat_playbook()

{'state_name': 'Región o estado geográfico del cliente (por ejemplo, GUATEMALA). Úsalo para hacer referencia a cobertura o disponibilidad local.',
 'previous_classification': 'Etiqueta comercial previa (por ejemplo, NEW_CLIENT, NOT_INTERESTED, NOT_EFFECTIVE). Ajusta el tono según el contexto: incorporación, retención o reactivación.',
 'previous_calls': 'Número de llamadas o interacciones previas del cliente. Un valor alto puede reflejar interés o fricción; adapta la interpretación según la dirección del SHAP.',
 'client_age': 'Edad del cliente en años. Evita sesgos demográficos; úsala solo para ajustar el tono de comunicación si es necesario.',
 'network_age_years': 'Años desde que el cliente se unió a la red o servicio. Una mayor antigüedad puede sugerir lealtad o una oportunidad de reactivación.',
 'banking': 'Indicador binario que señala si el cliente utiliza servicios bancarios. Si es afirmativo, enfatiza confianza y conveniencia; si no, mantén un tono neutral.',
 'arpu_90_days': 

# Get the global SHAP values

In [67]:
summary_df, features_block = helpers.top_global_features_from_drivers()

In [68]:
print(summary_df)

              feature  mean_abs_impact
0       plan_postpaid         0.327466
1            music_gb         0.302566
2          client_age         0.256492
3            contacts         0.250455
4   network_age_years         0.245034
5          minutes_in         0.201784
6        arpu_90_days         0.171417
7           apps_days         0.165527
8        charged_days         0.149878
9  start_using_months         0.146829


In [69]:
print(features_block)

- plan_postpaid: Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- music_gb: Cantidad de datos móviles (en GB) usados para música. Un valor cero representa oportunidad para ofertas de 'música sin consumo de datos'.
- client_age: Edad del cliente en años. Evita sesgos demográficos; úsala solo para ajustar el tono de comunicación si es necesario.
- contacts: Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- network_age_years: Años desde que el cliente se unió a la red o servicio. Una mayor antigüedad puede sugerir lealtad o una oportunidad de reactivación.
- minutes_in: Minutos de llamadas entrantes en el último periodo. Refleja el nivel de uso del servicio de voz.
- arpu_90_days: Ingreso promedio por usuario en los últimos 90 días. Un ARPU alto indica cliente activo o de alto valor; un ARPU bajo sugiere oportunidad d

# 🔹 **Level 1 — Global context prompt**


Includes an overview of the model and what the features represent:



*“The model uses 20 features such as ARPU, previous_calls, plan_postpaid, etc., to predict whether a prepaid user will make a purchase. The following SHAP analysis identifies the most influential features for this user.”*

👉 This gives the semantic model (Ollama or DeepSeek) the context of what the features are.

In [70]:
global_prompt = helpers.build_global_system_prompt_es()

### output

In [71]:
print(global_prompt)

Eres un asistente analítico para una empresa de telecomunicaciones. Tu función es ayudar a interpretar los principales drivers (valores SHAP) del modelo a nivel global y por cliente, en términos de negocio.

            Resumen Global de Drivers SHAP
            Estas son las variables globalmente más influyentes (TOP 10) y su significado de negocio:
            - plan_postpaid: Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- music_gb: Cantidad de datos móviles (en GB) usados para música. Un valor cero representa oportunidad para ofertas de 'música sin consumo de datos'.
- client_age: Edad del cliente en años. Evita sesgos demográficos; úsala solo para ajustar el tono de comunicación si es necesario.
- contacts: Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- network_age_years: Años desde que el cliente se unió 

### Prompt version 1

```bash
    global_prompt = """
    You are helping generate sales guidance for a prepaid telecom campaign.

    We have a machine learning model that predicts the probability that a customer will accept an offer (sale = 1).
    The model was trained on historical customer behavior and engagement indicators. 
    Higher score means the customer is more likely to buy if contacted.

    The model relies on multiple behavioral and account features. Below are the most influential features overall (averaged across customers), and what they represent:

    {features_block}

    Rules:
    - NEVER reveal internal model weights or math details.
    - NEVER invent personal/sensitive attributes not present in the features.
    - Keep tone helpful, respectful, and focused on value to customer.
    - You are allowed to explain *why the model thinks a segment is likely to buy*, in plain language.
        """.strip()
```

### prompt version 2

```bash
    global_prompt = f"""
    You are an expert sales advisor for a prepaid telecom campaign. Maximize conversions with sustainable offers aligned to each customer's real consumption.

    We use a machine learning model trained on historical behavior and engagement to estimate the probability of accepting an offer (sale=1). A higher score means higher likelihood if contacted.

    Most influential features overall (by mean absolute impact):
    {features_block}

    {rules_text}

    Policy:
        - Do NOT reveal internal model weights or math.
    - Do NOT invent personal/sensitive attributes beyond provided data.
    - Keep tone helpful, respectful, and value-focused.
     - You may explain drivers in plain language, but never mention “model”, “probability”, or “SHAP” in the final agent script.
    """.strip()
```

### prompt version 3

```bash

  if rules_text is None:
        rules_text = """
        Business Rules (apply consistently):
        1) Window: last 3 full months (M-1, M-2, M-3).
        2) Monthly ARPU = net revenue paid by the customer (top-ups, bundles, add-ons). Exclude freebies/bonuses, chargebacks, and adjustments.
        3) Eligibility: consumption > 0 in each of the 3 months, ARPU_3M_PROM ≥ Q80.00, and no commercial blocks.
        4) Offer mapping by ARPU_3M_PROM:
        • Q80.00–Q110.99 → PLAN_Q115
        • Q111.00–Q130.99 → PLAN_Q135
        • Q131.00–Q155.99 → PLAN_Q160
        • Q156.00–Q180.99 → PLAN_Q185
        • ≥ Q181.00       → PLAN_Q209
        5) Controlled upsell: if ARPU_3M_PROM is in the top 10% of its band and all three monthly ARPUs are ≥ 90% of the next band’s lower bound, offer the next band as an alternative.
        6) Downsell: on price objection, offer the minimum of the current band’s range (do not cross down a band unless affordability constraints are explicit).
        7) Messaging: emphasize benefits, keep price within the assigned band, and anchor value to actual spending.
        """.strip()

    global_prompt = f"""
    You are an expert sales advisor for a prepaid telecom campaign. Maximize conversions with sustainable offers aligned to each customer's real consumption.

    We use a machine learning model trained on historical behavior and engagement to estimate the probability of accepting an offer (sale=1). A higher score means higher likelihood if contacted.

    Most influential features overall (by mean absolute impact):
    {features_block}

    {rules_text}

    Policy:
        - Do NOT reveal internal model weights or math.
    - Do NOT invent personal/sensitive attributes beyond provided data.
    - Keep tone helpful, respectful, and value-focused.
     - You may explain drivers in plain language, but never mention “model”, “probability”, or “SHAP” in the final agent script.
    """.strip()

    return global_prompt
```

# 🔹 **Level 2 — Row-specific prompt (customized by customer)**


Here it uses the customer's SHAP drivers (their relevant features, values, and impacts):

In [75]:
idx = 42
row = df.loc[idx]

client_prompt = helpers.build_customer_prompt_summary(
    row=row,             # fila de df con 'proba'
    driver_list=row["drivers"],  # lista de dicts [{'feature','value','impact'}, ...]
)

### output

In [77]:
print(client_prompt)

Eres un analista de campañas de telecomunicaciones prepago.

        Analiza los factores más influyentes en la probabilidad de compra para un cliente individual, basándote en valores SHAP.
        El modelo predijo una probabilidad de aceptación del **65.4%**.

        A continuación se listan los principales *drivers* (variables) que explican esta predicción,
        ordenados por relevancia:

        - **contacts** (positivo (favorece contacto)): valor = 0.96. Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- **music_gb** (positivo (favorece contacto)): valor = 0.51. Cantidad de datos móviles (en GB) usados para música. Un valor cero representa oportunidad para ofertas de 'música sin consumo de datos'.
- **plan_postpaid** (positivo (favorece contacto)): valor = 0.69. Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad

### Prompt version 1

```bash
    prompt = f"""
    The model predicts a sale probability of {proba:.2%} for this prepaid customer.
    The most influential factors for this prediction were:
    {summary}

    Explain, in natural language, why these features may lead to this prediction.
    Provide a concise and interpretable summary.
    """
```

### prompt version 2

```bash
    prompt = f"""
    We are preparing a telemarketing/sales pitch for a prepaid mobile customer.

    Predicted probability of accepting the offer: {row['proba']:.2%}

    Relevant context for this customer:
    {context_block}

    The following factors were most influential in predicting that this customer is likely to accept an offer:
    {driver_block}

    Task:
    1. Explain, in plain language, why this customer might respond positively.
    2. Suggest how an agent should position the offer (tone, focus, what to mention).
    3. Keep it short and actionable, as guidance for a call center agent.
    4. Do NOT mention 'model', 'probability', 'algorithm', 'prediction', or 'SHAP'. Just speak as advice.
    """.strip()
```

### prompt version 3

```bash
    prompt = f"""
    [Customer Context]
    - Acceptance likelihood (score): {row.get('proba', float('nan')):.2%}
    - Attributes:
    {context_block}

    - Top influencing factors for this specific customer:
    {driver_block}

    [Task]
    Using ONLY the context above and the campaign rules from system prompt:
    1) Decide Eligibility: {{Yes/No}} and give a brief reason if "No".
    2) Select Suggested Band/Plan: {{PLAN_Q115|PLAN_Q135|PLAN_Q160|PLAN_Q185|PLAN_Q209}}.
    3) Provide Authorized offer range: {{Qxx.xx–Qyy.yy}}.
    4) Recommend an initial price within the authorized range: {{Qxx.xx}}. Justify in one line referencing recent spend.
    5) If applicable, propose an Upsell option (next band) with a one-line justification.

    Then produce a concise 3-line agent script:
    - Value: tie benefits to recent spend (“with what you already invest per month…”).
    - Price: keep within the assigned range.
    - Close: immediate activation, no contract, same line/top-ups.

    [Output Format]
    Eligibility: <Yes/No> (+ reason if No)
    Suggested Plan: <PLAN_Q115|PLAN_Q135|PLAN_Q160|PLAN_Q185|PLAN_Q209>
    Authorized range: <Qxx.xx–Qyy.yy>
    Recommended price: <Qxx.xx>  # one-line justification
    Upsell option: <plan_if_any>  # brief justification
    Script:
    “{name_value}, over the last 3 months you’ve invested about Q<arpu_3m_prom>/month.
    With the <plan_sugerido> plan you get more data/minutes for Q<precio_recomendado>, keeping your usual spend but with more value.
    Shall I confirm activation? It goes live today — no contract, and you keep your same line.”
    """.strip()
```

### prompt version 4


```bash
prompt = f"""
        [Customer Context]
        - Acceptance likelihood (score): {row.get('proba', float('nan')):.2%}
        - Attributes:
        {context_block}

        - Top influencing factors for this specific customer:
        {driver_block}

        [Task]
        Analyze these top 5 features from highest to lowest impact, using ONLY their values and impacts:

        1) Feature Analysis (Most Impactful):
        - Feature Context: Use the playbook description to understand what this raw metric means
        - Current Value: Interpret the specific value provided
        - Impact Direction: Consider if the impact is positive/negative for conversion
        - Action Insight: Determine specific actions based on this feature's influence

        2) Secondary Pattern:
        - Feature Context: Reference the playbook meaning for this raw metric
        - Value Analysis: Evaluate the concrete measurement provided
        - Impact Interpretation: Understand how this affects customer decision
        - Usage Pattern: Extract behavioral insights from this data point

        3) Supporting Evidence:
        - Feature Context: Apply the playbook definition
        - Data Point Analysis: Break down what the value indicates
        - Impact Assessment: Connect impact direction to customer behavior
        - Pattern Recognition: Identify relevant trends from this metric

        4) Behavioral Marker:
        - Feature Context: Consider the playbook explanation
        - Value Significance: Analyze what this measurement reveals
        - Impact Contribution: How this shapes customer response
        - Behavioral Insight: Extract actionable understanding

        5) Final Indicator:
        - Feature Context: Use the playbook to frame this metric
        - Value Review: What does this specific measurement tell us
        - Impact Role: How this influences overall likelihood
        - Opportunity Signal: Identify potential based on this data

        Then provide a data-driven synthesis:
        - Evidence Summary: Combine insights from ALL 5 features, with their specific values
        - Value Connection: Link each feature's impact to concrete benefits
        - Action Strategy: Propose steps based on the analyzed feature set

        [Output Format]
        Customer Analysis for {name_value}:

        Feature Impact Summary:
        1. Primary Driver: [Feature Name] Value: [Raw Metric] | Key Insight: [Based on Playbook]

        2. Secondary Driver: [Feature Name] |Value: [Raw Metric] | Key Insight: [Based on Playbook]

        3. Tertiary Driver: [Feature Name] | Value: [Raw Metric] | Key Insight: [Based on Playbook]

        4. Fourth Driver: [Feature Name] | Value: [Raw Metric] | Key Insight: [Based on Playbook]

        5. Fifth Driver: [Feature Name] | Value: [Raw Metric] | Key Insight: [Based on Playbook]

        Evidence-Based Summary:
        • Patterns: [Key findings from top 3 features]
        • Profile: [Customer behavior based on values]
        • Actions: [Data-supported recommendations]
        """.strip()
```

## Feature Engineering Conclusions

### Executive Summary

This notebook presents an advanced **Individualized Feature Engineering** system for telemarketing campaigns, where each customer has their own set of relevant characteristics with specific weights determined by SHAP values. Unlike traditional approaches that use a global set of features, this system dynamically adapts feature importance according to each client's individual profile.

### Individualized FE System Architecture

**1. Personalized Data Structure**
- **Individualized drivers**: Each row contains a 'drivers' column with client-specific characteristics
- **Structured format**: `[{"feature": "arpu_90_days", "value": 123.4, "impact": 0.05}, ...]`
- **Dynamic weights**: SHAP impacts vary by customer, reflecting the specific relevance of each characteristic
- **Adaptive flexibility**: No fixed set of global "top features" exists, but rather contextual adaptation

**2. Integrated Helpers System**
- **Helpers Class**: Centralizes processing logic and prompt generation
- **Specialized methods**: Dedicated functions for different levels of analysis
- **Code reusability**: Modular architecture for different use cases

### System Components Analysis

**1. Feature Playbook (Characteristics Dictionary)**
```python
helpers.get_feat_playbook()
```
- **Purpose**: Semantic mapping of technical characteristics to business insights
- **Functionality**: Translates raw metrics (e.g., arpu_90_days) to interpretable context
- **Added value**: Enables LLMs to understand the business meaning of each feature
- **Maintainability**: Centralizes definitions for consistency across the entire application

**2. Global SHAP Features Analysis**
```python
summary_df, features_block = helpers.top_global_features_from_drivers()
```
- **Intelligent aggregation**: Calculates average importance across all customers
- **Global ranking**: Identifies the most influential characteristics at macro level
- **Reference context**: Provides baseline for individual comparisons
- **Strategic insights**: Reveals general behavioral patterns in the customer base

### Multi-level Prompting System

#### **Level 1: Global System Context**
The global prompt establishes the reference framework for the language model:

**Global Prompt Evolution:**
- **Version 1**: Basic focus on model explainability
- **Version 2**: Incorporation of business rules and policies
- **Version 3**: Complete system with detailed business rules

**Key Components of Final Prompt:**
```bash
- System role: "Expert sales advisor for prepaid telecom campaign"
- Objective: "Maximize conversions with sustainable offers"
- Model context: Trained on historical behavior and engagement
- Most influential features: Global ranking by mean absolute impact
- Business rules: 3-month window, eligibility, offer mapping
- Policies: Don't reveal model weights, maintain respectful tone
```

#### **Level 2: Customer-Specific Prompts**
The system generates personalized prompts based on individual SHAP drivers:

**Customer Prompt Evolution:**
1. **Version 1**: Basic explanation of probability and factors
2. **Version 2**: Guidance for offer positioning and tone
3. **Version 3**: Eligibility analysis and structured scripts
4. **Version 4**: Detailed feature-by-feature analysis with actionable insights

**Final Prompt Structure (Version 4):**
```bash
- Customer Context: Acceptance probability and attributes
- Feature Analysis: Analysis of top 5 features with highest impact
- Impact Interpretation: Connection between values and customer decisions
- Evidence-Based Summary: Pattern synthesis and recommendations
```

### System Technical Insights

**1. Advanced Personalization**
- **Contextual adaptation**: Each customer receives analysis based on their own SHAP drivers
- **Dynamic relevance**: Important characteristics vary by individual
- **Improved precision**: Higher accuracy by considering customer-specific patterns

**2. Enhanced Interpretability**
- **Natural explanations**: Translates technical SHAP values to business language
- **Business context**: Connects metrics with actionable insights
- **Transparency**: Enables understanding the "why" behind each recommendation

**3. System Scalability**
- **Modular architecture**: Reusable helpers for different use cases
- **Efficient processing**: Optimized handling of nested drivers
- **Extensibility**: Easy incorporation of new features or business rules

### Strategic Implications

**1. For Call Center Operations**
- **Personalized scripts**: Each agent receives client-specific guidance
- **Higher conversion**: Messages adapted to individual prospect drivers
- **Improved efficiency**: Less preparation time with pre-calculated insights

**2. For Campaign Management**
- **Advanced segmentation**: Automatic identification of high-value profiles
- **Offer optimization**: Intelligent mapping based on historical spending patterns
- **Improved ROI**: Higher precision in targeting and positioning

**3. For Product Development**
- **Behavioral insights**: Deep understanding of adoption drivers
- **Feature validation**: Identification of most predictive characteristics
- **Model evolution**: Foundation for iterative system improvements

### Competitive Advantages of the Approach

**1. Versus Traditional Models**
- **Granularity**: Individual analysis vs. mass segmentation
- **Adaptability**: Dynamic weights vs. fixed characteristics
- **Precision**: Specific explanations vs. generalizations

**2. Versus Static Systems**
- **Continuous updating**: SHAP drivers reflect current patterns
- **Relevant context**: Important characteristics per specific customer
- **Flexibility**: Easy adaptation to business changes

### Technical Considerations and Limitations

**1. Computational Complexity**
- **Intensive processing**: SHAP values calculation for each customer
- **Storage**: Management of nested drivers per record
- **Scalability**: Considerations for large customer bases

**2. Data Quality**
- **Feature dependency**: System sensitive to input data quality
- **Updates**: Need for periodic refresh of SHAP drivers
- **Completeness**: Impact of missing values in individual drivers

**3. Interpretability vs. Complexity**
- **Balance**: Maintain explainability without sacrificing precision
- **Validation**: Need to verify explanation coherence
- **Training**: Requirement for end-user training

### Implementation Recommendations

**1. Gradual Deployment**
- **Controlled pilot**: Test with high-value customer subset
- **A/B Testing**: Compare against traditional methods
- **Continuous monitoring**: Track conversion metrics and feedback

**2. Operational Optimization**
- **Agent training**: Training in SHAP insights interpretation
- **Support tools**: Dashboards for driver visualization
- **Feedback loops**: Incorporation of results for model improvement

**3. System Evolution**
- **Feature engineering**: Incorporation of new predictive characteristics
- **Prompt refinement**: Iteration based on real performance
- **Integration**: Connection with CRM and campaign management systems

### Expected Impact

**Projected Success Metrics:**
- **Conversion Rate**: 15-25% increase vs. traditional methods
- **Customer Satisfaction**: Improvement in perceived offer relevance
- **Agent Efficiency**: Reduction in preparation time per call
- **Revenue per Campaign**: ROI optimization through higher targeting precision

This individualized Feature Engineering system represents a **qualitative leap** in telemarketing campaign personalization, combining the power of SHAP values with the interpretability of advanced prompting systems to create truly personalized and effective experiences for each individual customer.